## Imports

In [0]:
from pyspark.sql.functions import (
    col,
    lower,
    upper,
    trim,
    to_date,
    current_timestamp,
    row_number,
    lit,
    when,
    lpad
)
from pyspark.sql.window import Window

## Bronze table

In [0]:
bronze_df = spark.table("bronze_orders")
silver_customers = spark.table("silver_customers")

display(bronze_df.limit(10))
print(f"Bronze rows: {bronze_df.count():,}")

order_id,customer_id,order_date,order_status,order_amount,currency,sales_channel,created_at,updated_at,source_system
ORD000000001,CRM0039575,2023-07-17T16:16:05.996Z,PLACED,1133.55,AUD,WEB,2023-07-17T16:16:05.996Z,2023-07-20T16:16:05.996Z,ecommerce
ORD000000002,CRM0039575,2025-07-31T16:16:05.996Z,RETURNED,3470.62,AUD,STORE,2025-07-31T16:16:05.996Z,2025-08-21T16:16:05.996Z,ecommerce
ORD000000003,CRM0039575,2024-08-18T16:16:05.996Z,SHIPPED,3089.93,AUD,WEB,2024-08-18T16:16:05.996Z,2024-08-26T16:16:05.996Z,ecommerce
ORD000000004,CRM0039575,2024-09-09T16:16:05.996Z,RETURNED,1455.64,AUD,MOBILE,2024-09-09T16:16:05.996Z,2024-10-08T16:16:05.996Z,ecommerce
ORD000000005,CRM0039575,2025-05-22T16:16:05.996Z,SHIPPED,3885.04,AUD,STORE,2025-05-22T16:16:05.996Z,2025-06-04T16:16:05.996Z,ecommerce
ORD000000006,CRM0039575,2022-06-24T16:16:05.996Z,RETURNED,4223.83,AUD,MOBILE,2022-06-24T16:16:05.996Z,2022-07-14T16:16:05.996Z,ecommerce
ORD000000007,CRM0039575,2023-11-30T16:16:05.996Z,CANCELLED,2422.42,AUD,MOBILE,2023-11-30T16:16:05.996Z,2023-12-19T16:16:05.996Z,ecommerce
ORD000000008,CRM0000538,2021-10-01T16:16:05.996Z,SHIPPED,1123.5,AUD,STORE,2021-10-01T16:16:05.996Z,2021-10-18T16:16:05.996Z,ecommerce
ORD000000009,CRM0000538,2023-05-31T16:16:05.996Z,SHIPPED,2028.61,AUD,WEB,2023-05-31T16:16:05.996Z,2023-06-19T16:16:05.996Z,ecommerce
ORD000000010,CRM0000538,2022-06-10T16:16:05.996Z,DELIVERED,3382.58,AUD,STORE,2022-06-10T16:16:05.996Z,2022-07-03T16:16:05.996Z,ecommerce


Bronze rows: 484,628


## Silver transformation

In [0]:
VALID_STATUSES = [
    "PLACED",
    "PAID",
    "SHIPPED",
    "DELIVERED",
    "CANCELLED",
    "RETURNED"
]

VALID_CHANNELS = ["WEB", "MOBILE", "STORE"]

standardised_df = (
    bronze_df
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("order_status", upper(trim(col("order_status"))))
    .withColumn("currency", upper(trim(col("currency"))))
    .withColumn("sales_channel", upper(trim(col("sales_channel"))))
    .withColumn("created_at", to_date(col("created_at")))
    .withColumn("updated_at", to_date(col("updated_at")))
    .withColumn("_silver_processed_at", current_timestamp())
)
window_spec = Window.partitionBy("order_id").orderBy(col("updated_at").desc_nulls_last())
deduped_df = (
    standardised_df
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)
customer_lookup = (
    silver_customers
    .select("customer_id")
    .dropDuplicates(["customer_id"])
    .withColumn("customer_exists", lit(True))
)
orders_with_customer_check = (
    deduped_df
    .join(
        customer_lookup,
        on="customer_id",
        how="left"
    )
)
quality_df = (
    orders_with_customer_check
    .withColumn(
        "_has_future_updated_at",
        col("updated_at") > current_timestamp()
    )
    .withColumn(
        "_is_valid",
        when(col("order_id").isNull(), lit(False))
        .when(col("customer_id").isNull(), lit(False))
        .when(col("customer_exists").isNull(), lit(False))
        .when(~col("order_status").isin(VALID_STATUSES), lit(False))
        .when(col("sales_channel").isNull(), lit(False))
        .when(~col("sales_channel").isin(VALID_CHANNELS), lit(False))
        .when(col("order_amount")<=0, lit(False))
        .otherwise(lit(True))
    )
    .withColumn(
        "_rejection_reason",
        when(col("order_id").isNull(), lit("order_id is null"))
        .when(col("customer_id").isNull(), lit("customer_id is null"))
        .when(col("customer_exists").isNull(), lit("customer_id does not exist in silver customers"))
        .when(~col("order_status").isin(VALID_STATUSES), lit("invalid status"))
        .when(col("sales_channel").isNull(), lit("sales_channel is null"))
        .when(~col("sales_channel").isin(VALID_CHANNELS), lit("invalid sales_channel"))
        .when(col("order_amount")<=0, lit("order_amount must be greater than zero"))
        .otherwise(lit(None))
    )
)
valid_orders = quality_df.filter(col("_is_valid") == True)
rejected_orders = quality_df.filter(col("_is_valid") == False)


## Delta tables

In [0]:
(
    valid_orders
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_orders")
)

(
    rejected_orders
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("rejected_orders")
)

print("Silver Orders completed")
print(f"Bronze rows: {bronze_df.count():,}")
print(f"After deduplication: {deduped_df.count():,}")
print(f"Valid orders: {valid_orders.count():,}")
print(f"Rejected orders: {rejected_orders.count():,}")

Silver Orders completed
Bronze rows: 484,628
After deduplication: 479,830
Valid orders: 469,825
Rejected orders: 10,005


In [0]:
spark.sql("SHOW TABLES").show()

+--------+------------------+-----------+
|database|         tableName|isTemporary|
+--------+------------------+-----------+
| default|  bronze_addresses|      false|
| default|  bronze_customers|      false|
| default|     bronze_orders|      false|
| default|   bronze_payments|      false|
| default|rejected_addresses|      false|
| default|rejected_customers|      false|
| default|   rejected_orders|      false|
| default|  silver_addresses|      false|
| default|  silver_customers|      false|
| default|     silver_orders|      false|
+--------+------------------+-----------+

